In [1]:
# Regrid country/region masks to match population grid

In [2]:
import xarray as xr

In [3]:
# === Path config ===
POP_DIR = "/glade/work/awells/air_quality/SSP_pop/SSP2/"
MASKS_DIR = "/glade/work/awells/air_quality/BMR/masks/"

In [4]:
# === Population (i.e. target grid - 0.1x0.1) ===
population = xr.open_dataarray(f"{POP_DIR}ssp2_total_regrid_2000-2100.nc")

pop_lat = population.lat.values
pop_lon = population.lon.values

In [5]:
# === Country Mask ===
country_mask = xr.open_dataarray(f"{MASKS_DIR}/country/GBD_Country_Masks_0.10_newlabels.nc")

masks_lat = country_mask.lat.values
masks_lon = country_mask.lon.values

# Masks have slightly different lat/lon limits so we want to select the same points
# as the population data
masks_lat_mask = xr.ufuncs.logical_and(masks_lat <= pop_lat.max(), masks_lat >= pop_lat.min())
masks_lon_mask = xr.ufuncs.logical_and(masks_lon <= pop_lon.max(), masks_lon >= pop_lon.min())

# Match dimensions to population exactly
new_masks = country_mask.sel(lat=masks_lat_mask)  # longitudes are the same
masks_interp = new_masks.interp(lat=population.lat, lon=population.lon, method="linear")

masks_interp.to_netcdf(f"{MASKS_DIR}/country/GBD_Country_Masks_0.10_popgrid_newlabels.nc")

In [6]:
# === Region Mask ===
region_mask = xr.open_dataarray(f"{MASKS_DIR}/region/GBD_Region_Masks_0.10.nc")

masks_lat = region_mask.lat.values
masks_lon = region_mask.lon.values

# Masks have slightly different lat/lon limits so we want to select the same points
# as the population data
masks_lat_mask = xr.ufuncs.logical_and(masks_lat <= pop_lat.max(), masks_lat >= pop_lat.min())
masks_lon_mask = xr.ufuncs.logical_and(masks_lon <= pop_lon.max(), masks_lon >= pop_lon.min())

# Match dimensions to population exactly
new_masks = region_mask.sel(lat=masks_lat_mask)  # longitudes are the same
masks_interp = new_masks.interp(lat=population.lat, lon=population.lon, method="linear")

masks_interp.to_netcdf(f"{MASKS_DIR}/region/GBD_Region_Masks_0.10_popgrid.nc")